<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "omnicloudmask[legacy]==1.7.0" tacoreader transformers scipy

In [ ]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from IPython.display import display

from cloudband.colab.session import reload_package, start

reload_package("cloudband")

from cloudband.baselines import ocm
from cloudband.pipelines import phase2_eval
from cloudband.train.store import RunStore, ensure_shared_folder

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

BASELINE_DIR = Path("/content/drive/MyDrive/cloudband/results")
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER = Path("/content/drive/MyDrive/cloudband_treino")
ensure_shared_folder(DRIVE_FOLDER)
RESULTS_DIR = DRIVE_FOLDER / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

session = start(PROJECT_DIR, require_accelerator=True)
store = RunStore(
    results_dir=RESULTS_DIR,
    checkpoint_dir=DRIVE_FOLDER / "checkpoints",
    source_dir=Path(PROJECT_DIR) / "models",
)
store.wake()
print(f"published OCM results: {BASELINE_DIR}")
print(f"phase 2 shared folder: {DRIVE_FOLDER}")

In [ ]:
import getpass
import os

import requests

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face token (type 'Read' token, hidden): ").strip()

os.environ.pop("HF_TOKEN", None)
if token:
    reply = requests.get(
        "https://huggingface.co/api/whoami-v2",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if reply.status_code == 200:
        os.environ["HF_TOKEN"] = token
        print(f"token accepted for the account {reply.json().get('name', '?')}")
    else:
        print(f"token refused (HTTP {reply.status_code}); going on without it")
else:
    print("no token: requests use the shared anonymous quota")
del token

In [ ]:
tables = phase2_eval.load_test_tables()
print("test patches, L1C (primary):", len(tables["test"]))
print("test patches, L2A (secondary):", len(tables["test_l2a"]))

In [ ]:
cache = phase2_eval.build_test_cache(Path("/content/cloudband_cache"), tables)

## OmniCloudMask publicado: versões 1.0 a 4.0

In [ ]:
ocm.check_version()
published = phase2_eval.score_published(cache, tables, RESULTS_DIR)
print("published versions scored at L1C and L2A:", phase2_eval.PUBLISHED_VERSIONS)

Mesmo BOA por patch lendo do cache e lendo do arquivo remoto (avaliação anterior, só L1C): a diferença deve ser 0.

In [ ]:
phase2_eval.published_gaps(published, BASELINE_DIR)

Métricas do split inteiro (%) de cada versão publicada.

In [ ]:
phase2_eval.published_scores(published).round(2)

Tabela no formato do artigo do CloudSEN12 (BOA mediano por patch e faixas de PA e UA).

In [ ]:
phase2_eval.published_cloudsen12_tables(published)

## Modelos treinados: OCM (regnety + convnext) e Swin-T + UPerNet, 5 seeds

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

counts = phase2_eval.score_trained(cache, tables, store, RESULTS_DIR, DEVICE)

gaps = phase2_eval.legacy_gaps(counts["test"], RESULTS_DIR)
if not gaps.empty:
    display(gaps)

In [ ]:
combined = phase2_eval.with_published(counts, published)
reports = phase2_eval.build_reports(combined)
saved = phase2_eval.save_reports(reports, RESULTS_DIR)
print(f"{len(saved)} tables saved to {RESULTS_DIR}")

## Resultados no teste L1C (principal)

### Onde cada modelo fica
BOA do split inteiro: as quatro versões publicadas e os modelos treinados (cada seed e a média).

In [ ]:
phase2_eval.table_all_models(reports, "L1C").round(2)

In [ ]:
phase2_eval.show(phase2_eval.figure_all_models(reports, "L1C"), FIGURES_DIR, "l1c-all-models", display)

### O OCM treinado é melhor que o Swin treinado?
Diferença de BOA por ROI (média das seeds), com intervalo de 95%. Cada backbone do OCM sozinha mostra quanto vem do ensemble.

In [ ]:
phase2_eval.table_ocm_vs_swin(reports, "L1C").round(3)

In [ ]:
phase2_eval.show(phase2_eval.figure_ocm_vs_swin(reports, "L1C"), FIGURES_DIR, "l1c-ocm-vs-swin", display)

### Em quantos ROIs o OCM vence o Swin?
Distribuição da diferença OCM − Swin por ROI.

In [ ]:
phase2_eval.show(phase2_eval.figure_roi_wins(combined, "L1C"), FIGURES_DIR, "l1c-ocm-minus-swin-per-roi", display)

### Os modelos treinados contra cada versão publicada do OCM
Diferença de BOA por ROI do OCM e do Swin treinados para cada versão publicada, com intervalo de 95%.

In [ ]:
phase2_eval.table_vs_published(reports, "L1C").round(3)

In [ ]:
phase2_eval.show(phase2_eval.figure_vs_published(reports, "L1C"), FIGURES_DIR, "l1c-vs-published", display)

### Cada ROI: modelos treinados contra o OCM publicado 4.0
Um ponto por ROI; acima da diagonal, o modelo treinado foi melhor.

In [ ]:
phase2_eval.show(phase2_eval.figure_roi_vs_published(combined, "L1C"), FIGURES_DIR, "l1c-roi-vs-published-4", display)

## Resultados no teste L2A (secundário, nunca somado ao L1C)

### Onde cada modelo fica
BOA do split inteiro: as quatro versões publicadas e os modelos treinados (cada seed e a média).

In [ ]:
phase2_eval.table_all_models(reports, "L2A").round(2)

In [ ]:
phase2_eval.show(phase2_eval.figure_all_models(reports, "L2A"), FIGURES_DIR, "l2a-all-models", display)

### O OCM treinado é melhor que o Swin treinado?
Diferença de BOA por ROI (média das seeds), com intervalo de 95%. Cada backbone do OCM sozinha mostra quanto vem do ensemble.

In [ ]:
phase2_eval.table_ocm_vs_swin(reports, "L2A").round(3)

In [ ]:
phase2_eval.show(phase2_eval.figure_ocm_vs_swin(reports, "L2A"), FIGURES_DIR, "l2a-ocm-vs-swin", display)

### Em quantos ROIs o OCM vence o Swin?
Distribuição da diferença OCM − Swin por ROI.

In [ ]:
phase2_eval.show(phase2_eval.figure_roi_wins(combined, "L2A"), FIGURES_DIR, "l2a-ocm-minus-swin-per-roi", display)

### Os modelos treinados contra cada versão publicada do OCM
Diferença de BOA por ROI do OCM e do Swin treinados para cada versão publicada, com intervalo de 95%.

In [ ]:
phase2_eval.table_vs_published(reports, "L2A").round(3)

In [ ]:
phase2_eval.show(phase2_eval.figure_vs_published(reports, "L2A"), FIGURES_DIR, "l2a-vs-published", display)

### Cada ROI: modelos treinados contra o OCM publicado 4.0
Um ponto por ROI; acima da diagonal, o modelo treinado foi melhor.

In [ ]:
phase2_eval.show(phase2_eval.figure_roi_vs_published(combined, "L2A"), FIGURES_DIR, "l2a-roi-vs-published-4", display)

## L1C e L2A dão a mesma conclusão?
OCM − Swin por ROI nos dois níveis.

In [ ]:
print("OCM minus Swin has the same sign at L1C and L2A:", phase2_eval.level_agreement(reports))
phase2_eval.show(phase2_eval.figure_levels(reports), FIGURES_DIR, "l1c-vs-l2a-ocm-vs-swin", display)